# QDGrasp — Kaggle GPU Benchmark & Training

This notebook demonstrates how to install, verify, and run **QDGrasp** on Kaggle with GPU acceleration (NVIDIA T4 / P100).

### Features:
1. **Zero-overhead library installation**: standard `pip install -e .`
2. **Full test suite execution**: 310+ test cases across IK, transmission, physics rollouts, and scene synthesis.
3. **GPU Training**: Training cross-embodiment dexterous grasping models with PyTorch & CUDA.
4. **Full Verification Gates**: Automated audit of manifests, procedural object assets, and simulation rollouts.

In [ ]:
# Step 1: Clone repository (or navigate to uploaded code)
# If running in an uploaded dataset/code folder:
# %cd /kaggle/working/Dexgraspnet_custom

# Install dependencies and editable package
!pip install -q mujoco>=3.3.0 trimesh>=4.0.0 lightning>=2.6.0 safetensors typer rich pytest pytest-cov

In [ ]:
# Step 2: Verify GPU and Environment
import torch
import sys

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name   : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"PyTorch Version: {torch.__version__}")

In [ ]:
# Step 3: Run Full Pytest Verification Suite
!pytest -q

In [ ]:
# Step 4: Run Phase 3 Data Layer & Simulation Verification Gate
!python scripts/check_phase3.py

In [ ]:
# Step 5: Run GPU Model Training
from qdgrasp.api import QDGrasp

device = "cuda" if torch.cuda.is_available() else "cpu"
grasper = QDGrasp("qdgrasp-dummy-n.yaml", robot="leap_hand.yaml", seed=42)

result = grasper.train(
    "configs/data/dgn_open_tiny.yaml",
    device=device,
    max_steps=50,
    batch_size=16,
    learning_rate=1e-3,
    run_name="kaggle_gpu_train",
    project_dir="runs/kaggle",
)

print("Training Complete!")
print("Final Metrics:", result.metrics)